In [0]:
%python
from pyspark.sql import functions as F
import datetime as _dt

# =============================================================================
# PARAMETER EXTRACTION WITH DEFAULTS
# =============================================================================
try:
    arrival_date = dbutils.widgets.get("arrival_date")
except Exception:
    arrival_date = _dt.date.today().strftime("%Y-%m-%d")
try:
    catalog = dbutils.widgets.get("catalog")
except Exception:
    catalog = "travel_bookings"
try:
    schema = dbutils.widgets.get("schema")
except Exception:
    schema = "default"

# =============================================================================
# DQ RESULTS STORAGE SETUP
# =============================================================================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.ops")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.dq_results (
  business_date DATE,
  dataset STRING,
  check_name STRING,
  status STRING,
  constraint STRING,
  message STRING,
  recorded_at TIMESTAMP
) USING DELTA
""")

# =============================================================================
# SOURCE DATA PREPARATION
# =============================================================================
src = spark.table(f"{catalog}.bronze.customer_inc").where(
    F.col("business_date") == F.to_date(F.lit(arrival_date))
)

# =============================================================================
# DATA QUALITY CHECKS (single aggregation pass)
# =============================================================================
row_count = src.count()
null_checks = src.agg(
    F.count(F.when(F.col("customer_name").isNull(), 1)).alias("customer_name_nulls"),
    F.count(F.when(F.col("customer_address").isNull(), 1)).alias("customer_address_nulls"),
    F.count(F.when(F.col("email").isNull(), 1)).alias("email_nulls"),
).collect()[0]

# =============================================================================
# DQ RESULTS ASSEMBLY
# =============================================================================
now = _dt.datetime.now()

dq_results = [
    {
        "business_date": arrival_date,
        "dataset": "customer_inc",
        "check_name": "row_count",
        "status": "Success" if row_count > 0 else "Error",
        "constraint": "row_count > 0",
        "message": f"Row count: {row_count}",
        "recorded_at": now,
    },
    {
        "business_date": arrival_date,
        "dataset": "customer_inc",
        "check_name": "customer_name_not_null",
        "status": "Success" if null_checks.customer_name_nulls == 0 else "Error",
        "constraint": "customer_name IS NOT NULL",
        "message": f"Nulls: {null_checks.customer_name_nulls}",
        "recorded_at": now,
    },
    {
        "business_date": arrival_date,
        "dataset": "customer_inc",
        "check_name": "customer_address_not_null",
        "status": "Success" if null_checks.customer_address_nulls == 0 else "Error",
        "constraint": "customer_address IS NOT NULL",
        "message": f"Nulls: {null_checks.customer_address_nulls}",
        "recorded_at": now,
    },
    {
        "business_date": arrival_date,
        "dataset": "customer_inc",
        "check_name": "email_not_null",
        "status": "Success" if null_checks.email_nulls == 0 else "Error",
        "constraint": "email IS NOT NULL",
        "message": f"Nulls: {null_checks.email_nulls}",
        "recorded_at": now,
    },
]

# Cast types so they match the Delta table schema
dq_df = (
    spark.createDataFrame(dq_results)
    .withColumn("business_date", F.to_date(F.col("business_date")))
)

display(dq_df)

# =============================================================================
# DQ RESULTS LOGGING
# =============================================================================
dq_df.write.mode("append").saveAsTable(f"{catalog}.ops.dq_results")

# =============================================================================
# DQ VALIDATION AND ERROR HANDLING
# =============================================================================
if any(r["status"] == "Error" for r in dq_results):
    raise ValueError("DQ failed for customers")

print("Customer DQ passed")